# CI/CD Pipeline Automation (GitHub Actions + Python)

In [ ]:
# Day 50 — CI/CD Pipeline Automation
# We know pytest (Day 41), Docker (Day 46), Kubernetes (Day 47), and Secrets (Day 49).
# CI/CD ties them all together automatically on every `git push`!
#
# In MLOps, CI/CD is often called CT/CI/CD (Continuous Training + Integration + Delivery):
#   1. Lint & Unit Test (`pytest`)
#   2. Model / Data Gate (accuracy >= 0.85, Pandera validation passes)
#   3. Build & Push Docker image
#   4. Deploy to K8s & run Python Smoke Test
#
# Today we'll cover:
#   1. GitHub Actions YAML anatomy for Python ML projects
#   2. Writing Python CI/CD gate scripts (exit codes 0 vs 1)
#   3. Post-deploy Python smoke tests
#   4. Triggering GitHub Actions workflows via Python (`requests`)

### 1. GitHub Actions Workflow for MLOps

In [ ]:
import os
os.makedirs("cicd_demo", exist_ok=True)

gh_workflow = '''name: ML API CI/CD

on:
  push:
    branches: ["main"]
  pull_request:
    branches: ["main"]

jobs:
  test-and-gate:
    runs-on: ubuntu-latest
    strategy:
      matrix:
        python-version: ["3.10", "3.11"]
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: ${{ matrix.python-version }}
          cache: "pip"
      - run: pip install -r requirements.txt
      - run: pytest -v
      - run: python scripts/model_gate.py --min-acc 0.85

  build-and-deploy:
    needs: test-and-gate
    if: github.ref == 'refs/heads/main'
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - run: docker build -t ml-api:${{ github.sha }} .
      - run: python scripts/smoke_test.py --url http://localhost:8000
'''

with open("cicd_demo/ml_pipeline.yml", "w") as f:
    f.write(gh_workflow)

print("Created cicd_demo/ml_pipeline.yml")

### 2. Python CI/CD Gate & Smoke Test Scripts

In [ ]:
# CI/CD runners decide pass/fail based on EXIT CODES:
#   sys.exit(0) -> PASS (green checkmark)
#   sys.exit(1) -> FAIL (blocks merge / deployment!)

smoke_test = '''import sys
import time
import requests

def run_smoke_test(base_url: str, retries: int = 5) -> bool:
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(f"{base_url}/readyz", timeout=3)
            if r.status_code == 200:
                pred = requests.post(
                    f"{base_url}/predict",
                    json={"cpu": 85, "memory": 90, "disk": 70, "error_count": 25},
                    timeout=3
                )
                assert pred.status_code == 200
                print("Smoke test PASSED:", pred.json())
                return True
        except Exception as e:
            print(f"Attempt {attempt}/{retries} failed: {e}")
            time.sleep(2)
    return False

if __name__ == "__main__":
    ok = run_smoke_test("http://localhost:8000")
    sys.exit(0 if ok else 1)
'''

with open("cicd_demo/smoke_test.py", "w") as f:
    f.write(smoke_test)

print("Created cicd_demo/smoke_test.py")

### 3. Triggering Workflows Programmatically via GitHub API

In [ ]:
# Useful when an Evidently drift check or Airflow DAG wants to trigger a rebuild!

trigger_code = '''import os, requests

def trigger_github_workflow(owner: str, repo: str, workflow_id: str, ref: str = "main"):
    url = f"https://api.github.com/repos/{owner}/{repo}/actions/workflows/{workflow_id}/dispatches"
    headers = {
        "Authorization": f"Bearer {os.getenv('GITHUB_TOKEN')}",
        "Accept": "application/vnd.github+json"
    }
    resp = requests.post(url, headers=headers, json={"ref": ref}, timeout=10)
    resp.raise_for_status()
    print(f"Triggered {workflow_id} on {ref} (HTTP {resp.status_code})")
'''
print(trigger_code)

In [ ]:
# Practice Questions:
#
# 1. Why does `build-and-deploy` use `needs: test-and-gate` and `if: github.ref == 'refs/heads/main'`?
# 2. Why are `sys.exit(0)` and `sys.exit(1)` critical in CI/CD Python scripts?
# 3. How does `cache: 'pip'` speed up GitHub Actions workflows?

### DevOps / AI-MLOps Practice

In [ ]:
# Exercise 1: ML Model CI Gate Script
# Write `model_gate.py` that loads a model & test CSV, checks accuracy >= 0.85
# and p95 inference latency < 50ms, and exits 1 if either fails.

# Exercise 2: Automated Rollback Script
# Combine Day 47 (Kubernetes client) + Day 50 (Smoke test): if `run_smoke_test()`
# returns False, patch the deployment image back to the previous tag.

# Exercise 3: Complete GitHub Actions YAML
# Extend `ml_pipeline.yml` to upload `drift_report.html` as a workflow artifact
# using `actions/upload-artifact@v4`.